# Movie Recommendation System
This notebook demonstrates a content-based movie recommendation system using TF-IDF Vectorization and Cosine Similarity. It takes a movie title as input and suggests similar movies based on textual metadata (genres, keywords, cast, etc.).

## 1. Import Libraries
We will use `pandas` for data manipulation, and `scikit-learn` for text vectorization and computing similarity scores.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import difflib

import warnings
warnings.filterwarnings('ignore')

## 2. Load the Dataset
We use a publicly available movies dataset that contains relevant metadata for content-based filtering.

In [2]:
url = 'https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Movies%20Recommendation.csv'
df = pd.read_csv(url)

print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (4760, 21)


,Movie_ID,Movie_Title,Movie_Genre,Movie_Language,Movie_Budget,Movie_Popularity,Movie_Release_Date,Movie_Revenue,Movie_Runtime,Movie_Vote,...,Movie_Homepage,Movie_Keywords,Movie_Overview,Movie_Production_House,Movie_Production_Country,Movie_Spoken_Language,Movie_Tagline,Movie_Cast,Movie_Crew,Movie_Director
0,1,Four Rooms,Crime Comedy,en,4000000,22.876230,09-12-1995,4300000,98.0,6.5,...,NaN,hotel new year's eve witch bet hotel room,It's Ted the Bellhop's first night on the job....,"[{""name"": ""Miramax Films"", ""id"": 14}, {""name"":...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]",Twelve outrageous guests. Four scandalous requ...,Tim Roth Antonio Banderas Jennifer Beals Madon...,"[{'name': 'Allison Anders', 'gender': 1, 'depa...",Allison Anders
1,2,Star Wars,Adventure Action Science Fiction,en,11000000,126.393695,25-05-1977,775398007,121.0,8.1,...,http://www.starwars.com/films/star-wars-episod...,android galaxy hermit death star lightsaber,Princess Leia is captured and held hostage by ...,"[{""name"": ""Lucasfilm"", ""id"": 1}, {""name"": ""Twe...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]","A long time ago in a galaxy far, far away...",Mark Hamill Harrison Ford Carrie Fisher Peter ...,"[{'name': 'George Lucas', 'gender': 2, 'depart...",George Lucas
2,3,Finding Nemo,Animation Family,en,94000000,85.688789,30-05-2003,940335536,100.0,7.6,...,http://movies.disney.com/finding-nemo,father son relationship harbor underwater fish...,"Nemo, an adventurous young clownfish, is unexp...","[{""name"": ""Pixar Animation Studios"", ""id"": 3}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]","There are 3.7 trillion fish in the ocean, they...",Albert Brooks Ellen DeGeneres Alexander Gould ...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton
3,4,Forrest Gump,Comedy Drama Romance,en,55000000,138.133331,06-07-1994,677945399,142.0,8.2,...,NaN,vietnam veteran hippie mentally disabled runni...,A man with a low IQ has accomplished great thi...,"[{""name"": ""Paramount Pictures"", ""id"": 4}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]","The world will never be the same, once you've ...",Tom Hanks Robin Wright Gary Sinise Mykelti Wil...,"[{'name': 'Alan Silvestri', 'gender': 2, 'depa...",Robert Zemeckis
4,5,American Beauty,Drama,en,15000000,80.878605,15-09-1999,356296601,122.0,7.9,...,http://www.dreamworks.com/ab/,male nudity female nudity adultery midlife cri...,"Lester Burnham, a depressed suburban father in...","[{""name"": ""DreamWorks SKG"", ""id"": 27}, {""name""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...","[{""iso_639_1"": ""en"", ""name"": ""English""}]",Look closer.,Kevin Spacey Annette Bening Thora Birch Wes Be...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes


## 3. Data Cleaning and Feature Engineering
To compute similarity, we need to extract meaningful text features. We will combine `Movie_Genre`, `Movie_Keywords`, `Movie_Tagline`, `Movie_Cast`, and `Movie_Director` into a single string for each movie.

In [3]:
# Select relevant features for content-based filtering
features = ['Movie_Genre', 'Movie_Keywords', 'Movie_Tagline', 'Movie_Cast', 'Movie_Director']

# Fill missing values with empty strings
for feature in features:
    df[feature] = df[feature].fillna('')

# Combine features into a single string for each movie
def combine_features(row):
    return row['Movie_Genre'] + ' ' + row['Movie_Keywords'] + ' ' + row['Movie_Tagline'] + ' ' + row['Movie_Cast'] + ' ' + row['Movie_Director']

df['Combined_Features'] = df.apply(combine_features, axis=1)
df[['Movie_Title', 'Combined_Features']].head()

,Movie_Title,Combined_Features
0,Four Rooms,Crime Comedy hotel new year's eve witch bet ho...
1,Star Wars,Adventure Action Science Fiction android galax...
2,Finding Nemo,Animation Family father son relationship harbo...
3,Forrest Gump,Comedy Drama Romance vietnam veteran hippie me...
4,American Beauty,Drama male nudity female nudity adultery midli...


## 4. Text Vectorization (TF-IDF)
Machine learning algorithms work with numerical data. We use TF-IDF (Term Frequency-Inverse Document Frequency) to convert the combined text features into numerical vectors, assigning importance to words that are unique to specific movies.

In [4]:
vectorizer = TfidfVectorizer()
feature_vectors = vectorizer.fit_transform(df['Combined_Features'])
print(f"Feature Vectors Shape: {feature_vectors.shape}")

Feature Vectors Shape: (4760, 17258)


## 5. Calculate Cosine Similarity
Cosine similarity measures the cosine of the angle between two vectors. A higher score means the movies have more similar content.

In [5]:
similarity = cosine_similarity(feature_vectors)
print(f"Similarity Matrix Shape: {similarity.shape}")

Similarity Matrix Shape: (4760, 4760)


## 6. Recommendation Function
We create a function that takes a movie name, finds the closest match in our dataset (in case of minor typos), and returns the top 5 most similar movies.

In [ ]:
def recommend_movies(movie_name, df, similarity, num_recommendations=5):
    # Find the closest match to the movie name
    list_of_all_titles = df['Movie_Title'].tolist()
    find_close_match = difflib.get_close_matches(movie_name, list_of_all_titles)
    
    if not find_close_match:
        print("\n" + "="*50)
        print(f"No close match found for '{movie_name}' in the dataset.")
        print("="*50 + "\n")
        return
        
    close_match = find_close_match[0]
    
    # Find the index of the row directly
    movie_index = df.index[df['Movie_Title'] == close_match].tolist()[0]
    
    # Get similarity scores
    similarity_scores = list(enumerate(similarity[movie_index]))
    
    # Sort movies based on similarity
    sorted_similar_movies = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    print("\n" + "★"*50)
    print(f"  Top {num_recommendations} Recommendations for: '{close_match}'")
    print("★"*50 + "\n")
    
    # Print recommendations (skipping the first one as it is the movie itself)
    i = 1
    for movie in sorted_similar_movies[1:]:  # skip index 0
        index = movie[0]
        title_from_index = df.iloc[index]['Movie_Title']
        if i <= num_recommendations:
            print(f"  {i}. {title_from_index}")
            i += 1
        else:
            break
    print("\n" + "="*50 + "\n")

## 7. Test the Recommendation System
Let's test our recommendation engine with a popular sci-fi movie.

In [7]:
recommend_movies('Interstellar', df, similarity, num_recommendations=5)

recommend_movies('The Dark Knight', df, similarity, num_recommendations=5)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  Top 5 Recommendations for: 'Interstellar'
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

  1. The Dark Knight Rises
  2. The Matrix
  3. The Martian
  4. Dear Frankie
  5. Argo



★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  Top 5 Recommendations for: 'The Dark Knight'
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

  1. The Dark Knight Rises
  2. Batman Begins
  3. The Prestige
  4. Batman Returns
  5. Batman & Robin




## 8. Conclusion
The Movie Recommendation System successfully demonstrates the application of Natural Language Processing and machine learning to solve real-world content discovery problems. By leveraging TF-IDF vectorization and cosine similarity, we built an engine capable of recommending contextually relevant movies based entirely on metadata like cast, genres, and keywords without relying on user interaction history.